# Nemotron ColEmbed V2 — Colab smoke test (v2, consolidated)

**Purpose**: Verify the visual retriever loads on Colab GPU and produces multi-vector embeddings for a Sanofi PDF page.

**Phase**: 0 (environment setup)

---

## How to run (clean start, recommended)
1. **Runtime → Disconnect and delete runtime** (fresh VRAM)
2. **Runtime → Change runtime type → T4 GPU** (free tier) or L4 (if Pro compute units available)
3. Upload `Q1.pdf` to `/content/` via left sidebar folder icon
4. Left sidebar 🔑 icon → add `HF_TOKEN` secret
5. Run cells 1 → 7 in order

## Lessons baked in (from earlier attempts)
- **Poppler required** for `pdf2image` → apt-get install in cell 1
- **Use `torch.bfloat16`**, not fp16 (ViT internals are bf16-aware)
- **After loading, cast only float params/buffers to bf16** — preserve Long/Int indices (don't brute-cast everything)
- **Use `model.forward_images()` / `model.forward_queries()`**, not the generic `processor + model(**inputs)` pattern
- **fp32 OOMs on T4** (~12 GB needed, 14.5 GB available — too tight)

## Model options
| Model ID | Params | ViDoRe V3 rank | Runtime |
|---|---|---|---|
| **`nvidia/llama-nemotron-colembed-vl-3b-v2`** ★ default | 3B | 6th | T4 (free) |
| `nvidia/nemotron-colembed-vl-4b-v2` | 4B | 3rd | L4 (Pro) |
| `nvidia/nemotron-colembed-vl-8b-v2` | 8B | **1st** | L4 / A100 |

Default here is 3B for T4 compatibility. Swap `MODEL_ID` below if you have Pro compute.

## 1. Install system + Python dependencies

In [ ]:
!apt-get install -y -qq poppler-utils
!pip install -q transformers accelerate einops sentencepiece pypdfium2 pdf2image Pillow
!pip install -q pyngrok fastapi uvicorn nest-asyncio

## 2. HuggingFace authentication

Token at https://huggingface.co/settings/tokens (Classic → Read is fine).
Nemotron ColEmbed V2 is NOT gated — license is CC-BY-NC-4.0 (non-commercial). No Agree button click needed.

In [ ]:
from huggingface_hub import login
from google.colab import userdata
import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
login(token=os.environ['HF_TOKEN'])

## 3. Load Nemotron ColEmbed V2 (bf16 + safe cast)

In [ ]:
import torch
from transformers import AutoModel

# ─── Choose one ────────────────────────────────────────
MODEL_ID = 'nvidia/llama-nemotron-colembed-vl-3b-v2'     # ★ default, T4-friendly (6th on ViDoRe V3)
# MODEL_ID = 'nvidia/nemotron-colembed-vl-4b-v2'        # 3rd, needs L4
# MODEL_ID = 'nvidia/nemotron-colembed-vl-8b-v2'        # 1st (SOTA), needs L4 / A100
# ───────────────────────────────────────────────────────

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

model = AutoModel.from_pretrained(
    MODEL_ID,
    device_map='cuda',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
).eval()

# Force all FLOAT tensors to bf16 so ViT sub-module matches language model.
# DO NOT cast integer tensors (they're indices — must stay Long/Int).
FLOAT_DTYPES = (torch.float32, torch.float16, torch.bfloat16, torch.float64)
for p in model.parameters():
    if p.dtype in FLOAT_DTYPES:
        p.data = p.data.to(torch.bfloat16)
for b in model.buffers():
    if b.dtype in FLOAT_DTYPES:
        b.data = b.data.to(torch.bfloat16)

print(f'\nModel loaded. Params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B')
print(f'main param dtype: {next(model.parameters()).dtype}')
# Quick sub-module dtype check
for name, m in model.named_children():
    first_p = next(m.parameters(), None)
    if first_p is not None:
        print(f'  {name}: {first_p.dtype}')

## 4. Sample PDF → image

Upload `Q1.pdf` to `/content/` first (left sidebar folder icon → upload).

In [ ]:
from pdf2image import convert_from_path

PDF_PATH = '/content/Q1.pdf'
PAGE_NUM = 1   # 1-indexed

pages = convert_from_path(PDF_PATH, first_page=PAGE_NUM, last_page=PAGE_NUM, dpi=150)
img = pages[0]
print(f'Page {PAGE_NUM}: {img.size} (w, h)')
img

## 5. Embed the page (image)

Nemotron ColEmbed uses dedicated methods, not `model(**processor_inputs)`:
- `model.forward_images(images, batch_size=...)` for pages
- `model.forward_queries(queries, batch_size=...)` for text queries
- `model.get_scores(q_emb, d_emb)` for MaxSim similarity

Expected shape per page: `[num_tokens, hidden_dim]` (hidden_dim = 3072 for 3B-v2).

In [ ]:
images = [img]

with torch.no_grad():
    image_embeddings = model.forward_images(images, batch_size=1)

print(f'type: {type(image_embeddings)}')
if isinstance(image_embeddings, list):
    emb = image_embeddings[0]
else:
    emb = image_embeddings[0] if image_embeddings.dim() == 3 else image_embeddings

print(f'Embedding shape: {emb.shape}')
print(f'dtype: {emb.dtype}, device: {emb.device}')

## 6. Query + similarity (MaxSim via `get_scores`)

In [ ]:
queries = ['What was Dupixent sales in Q1 2025?']

with torch.no_grad():
    query_embeddings = model.forward_queries(queries, batch_size=1)

q_emb = query_embeddings[0] if isinstance(query_embeddings, list) else query_embeddings
print(f'Query emb shape: {q_emb.shape if hasattr(q_emb, "shape") else len(q_emb)}')

scores = model.get_scores(query_embeddings, image_embeddings)
print(f'Similarity score: {scores}')

## 7. Acceptance checklist (Phase 0)

- [ ] Cell 3: model loaded, `main param dtype: torch.bfloat16`, all sub-modules `bfloat16`
- [ ] Cell 4: PDF page rendered as PIL image (visible preview)
- [ ] Cell 5: Embedding shape `[N, 3072]` where N is ~hundreds to a few thousand
- [ ] Cell 6: Similarity score is a finite positive number

All 4 checks pass → Phase 0 Nemotron component cleared.

## Troubleshooting quick reference
| Error | Fix |
|---|---|
| `pdfinfo not found` | Cell 1 apt-get install poppler-utils wasn't run |
| `Half vs Float dtype mismatch` | Cell 3 FLOAT_DTYPES cast loop skipped |
| `embedding(): ... got CUDABFloat16Type` | Cell 3 cast corrupted integer buffers (the `if p.dtype in FLOAT_DTYPES` check missing) |
| `CUDA OOM during load` | Previous model still in VRAM → Runtime → **Disconnect and delete runtime** |
| `AttributeError: ... forward_images` | MODEL_ID is wrong / not a ColEmbed model |